# FEATURE SCALING

````markdown


Feature scaling changes the **numerical scale of features** so that features with different ranges become comparable.

Example:

```text
Age       → 18 to 80
Salary    → 20,000 to 2,00,000
````

Without scaling, features with larger numerical values can have a greater influence on algorithms that depend on distances, magnitudes, or gradients.

### Important for algorithms such as:

* KNN
* K-Means
* SVM
* PCA
* Neural Networks
* Linear Regression with regularization
* Logistic Regression with regularization
* Gradient-based algorithms

### Usually Less Important For:

* Decision Trees
* Random Forest
* Other tree-based models

Tree-based models generally split features based on thresholds rather than distances or feature magnitudes.

---

## 1. STANDARDIZATION

**Standardization** transforms a feature so that it has approximately:

```text
Mean = 0
Standard Deviation = 1
```

### Formula

```text
z = (x - mean) / standard deviation
```

### Python

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df["Age_scaled"] = scaler.fit_transform(
    df[["Age"]]
)
```

Standardization does **not** force values into a fixed range such as 0 to 1.

It is commonly used when features have different scales and the distribution is reasonably suitable for standardization.

### 🧠 Memory

> **Standardization → Mean 0, Standard Deviation 1**

---

## 2. NORMALIZATION / MIN-MAX SCALING

**Min-Max Scaling** transforms values to a specified range.

The most common range is:

```text
0 to 1
```

### Formula

```text
x_scaled = (x - min) / (max - min)
```

### Python

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["Age_scaled"] = scaler.fit_transform(
    df[["Age"]]
)
```

For the default range:

```text
Minimum → 0
Maximum → 1
```

### Important

Min-Max Scaling is **sensitive to outliers**.

If one feature contains an extreme value, the minimum and maximum can be heavily affected, causing most other values to become compressed into a small range.

### 🧠 Memory

> **Min-Max Scaling → Usually 0 to 1**

---

## 3. ROBUST SCALING

**Robust Scaling** uses the:

* Median
* Interquartile Range (IQR)

instead of:

* Mean
* Standard Deviation

### Formula

```text
x_scaled = (x - median) / IQR
```

where:

```text
IQR = Q3 - Q1
```

Because median and IQR are less affected by extreme values, Robust Scaling is **more resistant to outliers**.

### Python

```python
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

df["Salary_scaled"] = scaler.fit_transform(
    df[["Salary"]]
)
```

### Useful When

* Significant outliers are present
* Data is skewed
* Median and IQR better represent the feature

### 🧠 Memory

> **RobustScaler → Median + IQR → More resistant to outliers**

---

## 4. MAXABS SCALING

**MaxAbs Scaling** divides each value by the maximum absolute value of the feature.

### Formula

```text
x_scaled = x / max(|x|)
```

It usually maps values approximately into:

```text
-1 to 1
```

depending on the original values.

### Python

```python
from sklearn.preprocessing import MaxAbsScaler

scaler = MaxAbsScaler()

df["Feature_scaled"] = scaler.fit_transform(
    df[["Feature"]]
)
```

### Advantages

* Preserves zero
* Preserves the sign of values
* Useful for sparse data
* Does not center the data
* Does not destroy sparsity by subtracting the mean

### Example

```text
Original:
-10
  0
  5
 10
```

After MaxAbs Scaling:

```text
-1
 0
 0.5
 1
```

### 🧠 Memory

> **MaxAbsScaler → Divide by maximum absolute value → Useful for sparse data**

---

# COMPARISON

| Scaler             | Main Idea                        | Outlier Sensitivity | Typical Range         |
| ------------------ | -------------------------------- | ------------------- | --------------------- |
| **StandardScaler** | Mean = 0, Std = 1                | Sensitive           | No fixed range        |
| **MinMaxScaler**   | Uses minimum and maximum         | Sensitive           | Usually 0 to 1        |
| **RobustScaler**   | Uses median and IQR              | More resistant      | No fixed range        |
| **MaxAbsScaler**   | Divide by maximum absolute value | Sensitive           | Approximately -1 to 1 |

---

# CHOOSING A SCALER

### No major outliers

```text
StandardScaler
```

### Need a fixed range

```text
MinMaxScaler
```

### Significant outliers

```text
RobustScaler
```

### Sparse data

```text
MaxAbsScaler
```

These are general guidelines. The best scaler depends on the dataset, feature distribution, and ML algorithm.

---

## IMPORTANT ML RULE: FIT ONLY ON TRAINING DATA ⚠️

The scaler **learns parameters from the data**.

For example:

* StandardScaler learns mean and standard deviation.
* MinMaxScaler learns minimum and maximum.
* RobustScaler learns median and IQR.
* MaxAbsScaler learns maximum absolute value.

Therefore, the scaler should be fitted **only on training data**.

### Correct

```python
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)
```

### Incorrect

```python
X_test_scaled = scaler.fit_transform(X_test)
```

Do not fit the scaler separately on test data.

---

## WHY?

If the scaler learns parameters from the test data, information from the test set is used during preprocessing.

This can cause **data leakage**.

Correct workflow:

```text
Full Dataset
     ↓
Train / Test Split
     ↓
     ┌───────────────┐
     ↓               ↓
Training Data     Test Data
     ↓               ↓
fit_transform      transform
     ↓               ↓
Scaled Train      Scaled Test
```

### Key Rule

> **Fit on training data → Transform training data → Transform test data using the same fitted scaler.**

---

## FEATURE SCALING VS NORMALIZATION

In machine learning discussions, the word **normalization** is sometimes used broadly to mean feature scaling.

However, specifically:

```text
Standardization
→ Mean 0, Standard Deviation 1
```

```text
Min-Max Scaling
→ Maps values to a chosen range, commonly 0 to 1
```

So when discussing preprocessing precisely, it is better to name the exact scaling method being used.

---

## 🧠 QUICK MEMORY

```text
StandardScaler
→ Mean 0, Std 1
```

```text
MinMaxScaler
→ Usually 0 to 1
```

```text
RobustScaler
→ Median + IQR
→ Outlier resistant
```

```text
MaxAbsScaler
→ Divide by maximum absolute value
→ Preserves zero and sign
→ Good for sparse data
```

### Final Rule

> **Choose the scaler based on feature distribution, outliers, sparsity, and the requirements of the ML algorithm.**

```
```


In [2]:
import pandas as pd

df = pd.DataFrame({
    "Age": [18, 22, 25, 30, 35, 40, 45, 50],

    "Salary": [
        20000, 25000, 30000, 35000,
        40000, 50000, 60000, 500000
    ],

    "Score": [
        -100, -50, -20, 0,
        20, 50, 80, 100
    ]
})

In [3]:
from sklearn.preprocessing import StandardScaler

Scaler = StandardScaler()

df['Age_Scaled_Standard'] = Scaler.fit_transform(df[['Age']])

In [9]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df['Age_Scaled_MinMax'] = scaler.fit_transform(df[['Age']])
df['Age'].skew()

np.float64(0.18243219565445862)

In [10]:
# Age is almost close to normal distribution so we can use the StandardScaler()
df

,Age,Salary,Score,Age_Scaled_Standard,Age_Scaled_MinMax
0,18,20000,-100,-1.422152,0.00000
1,22,25000,-50,-1.046046,0.12500
2,25,30000,-20,-0.763966,0.21875
3,30,35000,0,-0.293833,0.37500
4,35,40000,20,0.176300,0.53125
5,40,50000,50,0.646433,0.68750
6,45,60000,80,1.116566,0.84375
7,50,500000,100,1.586699,1.00000


In [11]:
# for salary there is an outlier 500000 so data is right side skewed s
# so we preferring robust Scaler since its using IQR method which is robust to outliers

from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()

df["Salary_scaled_Robust"] = scaler.fit_transform(
    df[["Salary"]]
)

In [12]:
df

,Age,Salary,Score,Age_Scaled_Standard,Age_Scaled_MinMax,Salary_scaled_Robust
0,18,20000,-100,-1.422152,0.00000,-0.736842
1,22,25000,-50,-1.046046,0.12500,-0.526316
2,25,30000,-20,-0.763966,0.21875,-0.315789
3,30,35000,0,-0.293833,0.37500,-0.105263
4,35,40000,20,0.176300,0.53125,0.105263
5,40,50000,50,0.646433,0.68750,0.526316
6,45,60000,80,1.116566,0.84375,0.947368
7,50,500000,100,1.586699,1.00000,19.473684


In [15]:
# for score we can use the maxabs scaler
#  we have to preserve the values sign
from sklearn.preprocessing import MaxAbsScaler
scaler = MaxAbsScaler()

df['Score_MaxAbs'] = scaler.fit_transform(df[['Score']])

In [16]:
df

,Age,Salary,Score,Age_Scaled_Standard,Age_Scaled_MinMax,Salary_scaled_Robust,Score_MaxAbs
0,18,20000,-100,-1.422152,0.00000,-0.736842,-1.0
1,22,25000,-50,-1.046046,0.12500,-0.526316,-0.5
2,25,30000,-20,-0.763966,0.21875,-0.315789,-0.2
3,30,35000,0,-0.293833,0.37500,-0.105263,0.0
4,35,40000,20,0.176300,0.53125,0.105263,0.2
5,40,50000,50,0.646433,0.68750,0.526316,0.5
6,45,60000,80,1.116566,0.84375,0.947368,0.8
7,50,500000,100,1.586699,1.00000,19.473684,1.0
